# Memory demo: Calculator agent

Same agent, same questions, run twice:

1. **Without memory**: every call starts from a blank slate.
2. **With memory**: a checkpointer saves the conversation, so "that" and "it" make sense.

The questions are deliberately chained ("multiply *that* by 5"), so they only work if the agent remembers the previous answer.

## Setup

In [1]:
%%capture
%pip install --upgrade langgraph langchain langchain-aws langchain-community
%pip install --upgrade python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Load AWS credentials from .env (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION)
# boto3 / langchain-aws pick them up from the environment automatically.
load_dotenv()


True

## Calculator tools

In [3]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers: a + b."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a: a - b."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers: a * b."""
    return a / b

@tool
def divide(a: float, b: float) -> str | float:
    """Divide a by b: a / b."""
    if b == 0:
        return "Error: cannot divide by zero"
    return a / b

tools = [add, subtract, multiply, divide]

In [4]:
SYSTEM_PROMPT = """You are a calculator assistant. 

Rules:
- ALWAYS use the tools (add, subtract, multiply, divide) for arithmetic. Never calculate with your own knowledge. Use only the tool results exactly as returned.
- Call ONE tool at a time.
- If an expression requires multiple operations, execute them one by one. Do not provide a conversational text answer until ALL steps are completely calculated via tools.
- If the user references 'that', 'it', or 'the result', always use the result of the previous tool execution in the conversation.
- If you do not know which number the user means, ask them. Do not guess.
- Once all calculations for the user's request are completely finished, reply with a short sentence containing the final answer."""


In [5]:
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse

# temperature=0 keeps tool calling stable for a demo
chat = ChatBedrockConverse(
    model="mistral.mistral-large-3-675b-instruct",
    temperature=1,
)


## Helper

`ask()` sends one question and prints the answer. It also prints **how many messages the model received**. This number shows the difference between the two agents.

In [6]:
def ask(agent, question, thread_id=None):
    """Send one question. Pass thread_id only for an agent that has memory."""
    config = {"configurable": {"thread_id": thread_id}} if thread_id else None

    print(f"\n>>> {question}   (thread_id={thread_id})")
    last = None
    for step in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config,
        stream_mode="values",
    ):
        last = step

    # tool calls made in THIS turn only (everything after the last human message)
    msgs = last["messages"]
    print(msgs)
    turn = msgs[max(i for i, m in enumerate(msgs) if m.type == "human"):]
    for m in turn:
        for call in getattr(m, "tool_calls", []) or []:
            print(f"    tool call: {call['name']}({call['args']})")

    print(f"    messages in the agent's state: {len(last['messages'])}")
    print(f"    ANSWER: {last['messages'][-1].content}")

---
# Part 1: Without memory

No `checkpointer`. Every `ask()` starts with only the single question we send.

In [7]:
agent_no_memory = create_agent(
    model=chat,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

In [17]:
ask(agent_no_memory, "What is 12 plus 8?")


>>> What is 12 plus 8?   (thread_id=None)
[HumanMessage(content='What is 12 plus 8?', additional_kwargs={}, response_metadata={}, id='a8e0b373-2d2e-49ce-b812-f8af0dc16aff'), AIMessage(content=[{'type': 'tool_use', 'name': 'add', 'input': {'a': 12, 'b': 8}, 'id': 'tooluse_449mgKPUkY1MH9Befa6MAq'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '4dae6273-937a-4cf7-a7c8-42d5ffa68387', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 00:57:31 GMT', 'content-type': 'application/json', 'content-length': '287', 'connection': 'keep-alive', 'x-amzn-requestid': '4dae6273-937a-4cf7-a7c8-42d5ffa68387'}, 'RetryAttempts': 1}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [532]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2f5-3323-70d1-a825-86ccce517ec2-0', tool_calls=[{'name': 'add', 'args': {'a': 12, 'b': 8}, 'id': 'tooluse_449mgKPUkY1MH9Befa6MAq', 'type': 'tool_call'}], invalid_to

In [18]:
ask(agent_no_memory, "multiply that by 5")


>>> multiply that by 5   (thread_id=None)
[HumanMessage(content='multiply that by 5', additional_kwargs={}, response_metadata={}, id='da16b6ef-41db-4b93-af13-c3ced30b2fc5'), AIMessage(content='It seems you referred to "that," but I don\'t have a previous result yet. Could you clarify which number you\'d like me to multiply by 5?', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'c13566b4-dcfc-415a-99ac-08c38848b791', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 00:58:41 GMT', 'content-type': 'application/json', 'content-length': '341', 'connection': 'keep-alive', 'x-amzn-requestid': 'c13566b4-dcfc-415a-99ac-08c38848b791'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [537]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2f6-db7c-7892-8d51-03252d3197bc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 453, 'output_tokens': 34

### What happened without memory

- Question 1 works: `add(12, 8)` gives 20.
- Question 2 fails. The agent never saw question 1, so it does not know what "that" is. It asks which number you mean, or guesses.

Each call was a **brand new conversation**: the message count never grows past one question, its tool call and the answer.

---
# Part 2: With memory

The only code changes:
1. `checkpointer=InMemorySaver()` in `create_agent(...)` turns saving on.
2. `thread_id` in each call says which saved conversation to continue.

In [19]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = create_agent(
    model=chat,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

In [20]:
ask(agent_with_memory, "What is 12 plus 8?", thread_id="calc-1")


>>> What is 12 plus 8?   (thread_id=calc-1)
[HumanMessage(content='What is 12 plus 8?', additional_kwargs={}, response_metadata={}, id='d5246cbe-4838-40c3-b6dd-c9a91833f37e'), AIMessage(content=[{'type': 'tool_use', 'name': 'add', 'input': {'a': 12, 'b': 8}, 'id': 'tooluse_01FlC9LrMWzuAV6hMLR7oo'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'fe06e319-bc0c-4ac7-b7df-710c030fcded', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 01:00:11 GMT', 'content-type': 'application/json', 'content-length': '287', 'connection': 'keep-alive', 'x-amzn-requestid': 'fe06e319-bc0c-4ac7-b7df-710c030fcded'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [289]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2f8-3d50-7182-876f-2a6a2bdfb121-0', tool_calls=[{'name': 'add', 'args': {'a': 12, 'b': 8}, 'id': 'tooluse_01FlC9LrMWzuAV6hMLR7oo', 'type': 'tool_call'}], invalid_

In [21]:
ask(agent_with_memory, "Multiply that by 5", thread_id="calc-1")


>>> Multiply that by 5   (thread_id=calc-1)
[HumanMessage(content='What is 12 plus 8?', additional_kwargs={}, response_metadata={}, id='d5246cbe-4838-40c3-b6dd-c9a91833f37e'), AIMessage(content=[{'type': 'tool_use', 'name': 'add', 'input': {'a': 12, 'b': 8}, 'id': 'tooluse_01FlC9LrMWzuAV6hMLR7oo'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'fe06e319-bc0c-4ac7-b7df-710c030fcded', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 01:00:11 GMT', 'content-type': 'application/json', 'content-length': '287', 'connection': 'keep-alive', 'x-amzn-requestid': 'fe06e319-bc0c-4ac7-b7df-710c030fcded'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [289]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2f8-3d50-7182-876f-2a6a2bdfb121-0', tool_calls=[{'name': 'add', 'args': {'a': 12, 'b': 8}, 'id': 'tooluse_01FlC9LrMWzuAV6hMLR7oo', 'type': 'tool_call'}], invalid_

In [24]:
ask(agent_with_memory, "Now divide it by 4", thread_id="calc-1")


>>> Now divide it by 4   (thread_id=calc-1)
[HumanMessage(content='What is 12 plus 8?', additional_kwargs={}, response_metadata={}, id='d5246cbe-4838-40c3-b6dd-c9a91833f37e'), AIMessage(content=[{'type': 'tool_use', 'name': 'add', 'input': {'a': 12, 'b': 8}, 'id': 'tooluse_01FlC9LrMWzuAV6hMLR7oo'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'fe06e319-bc0c-4ac7-b7df-710c030fcded', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 01:00:11 GMT', 'content-type': 'application/json', 'content-length': '287', 'connection': 'keep-alive', 'x-amzn-requestid': 'fe06e319-bc0c-4ac7-b7df-710c030fcded'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [289]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2f8-3d50-7182-876f-2a6a2bdfb121-0', tool_calls=[{'name': 'add', 'args': {'a': 12, 'b': 8}, 'id': 'tooluse_01FlC9LrMWzuAV6hMLR7oo', 'type': 'tool_call'}], invalid_

In [14]:
ask(agent_with_memory, "Subtract 5 from the result", thread_id="calc-1")


>>> Subtract 5 from the result   (thread_id=calc-1)
[HumanMessage(content='What is 12 plus 8?', additional_kwargs={}, response_metadata={}, id='a8ecee7c-4454-472e-8a82-97657a6b1e0d'), AIMessage(content=[{'type': 'tool_use', 'name': 'add', 'input': {'a': 12, 'b': 8}, 'id': 'tooluse_miXIqI52DnjL5N1h9l3tje'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'e24f1213-aa50-42c0-ad15-658bed11c486', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 00:35:09 GMT', 'content-type': 'application/json', 'content-length': '287', 'connection': 'keep-alive', 'x-amzn-requestid': 'e24f1213-aa50-42c0-ad15-658bed11c486'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [356]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2e1-527f-76b3-abaf-2b1a799e7985-0', tool_calls=[{'name': 'add', 'args': {'a': 12, 'b': 8}, 'id': 'tooluse_miXIqI52DnjL5N1h9l3tje', 'type': 'tool_call'}], 

### What happened with memory

- "that" resolved to 20, so `multiply(20, 5)` gives 100.
- "it" resolved to 100, so `divide(100, 4)` gives 25.
- "the result" resolved to 25, so `subtract(25, 5)` gives 20.
- The message count **keeps growing**, because each call loads the saved history, adds the new question and saves it again.

## Look inside the memory

In [27]:
state = agent_with_memory.get_state({"configurable": {"thread_id": "calc-1"}})

for m in state.values["messages"]:
    if m.type == "ai" and m.tool_calls:
        c = m.tool_calls[0]
        print(f"AI    -> tool call {c['name']}({c['args']})")
    else:
        print(f"{m.type.upper():5} -> {str(m.content)[:80]}")

HUMAN -> What is 12 plus 8?
AI    -> tool call add({'a': 12, 'b': 8})
TOOL  -> 20.0
AI    -> The final answer is **20**.
HUMAN -> Multiply that by 5
AI    -> tool call multiply({'a': 20, 'b': 5})
TOOL  -> 4.0
AI    -> The final answer is **100**.
HUMAN -> Now divide it by 4
AI    -> tool call divide({'a': 100, 'b': 4})
TOOL  -> 25.0
AI    -> The final answer is **25**.
HUMAN -> Now divide it by 4
AI    -> tool call divide({'a': 25, 'b': 4})
TOOL  -> 6.25
AI    -> The final answer is **6.25**.


## Different `thread_id` = different conversation

In [16]:
ask(agent_with_memory, "Multiply that by 5", thread_id="calc-2")


>>> Multiply that by 5   (thread_id=calc-2)
[HumanMessage(content='Multiply that by 5', additional_kwargs={}, response_metadata={}, id='182a2087-7bf7-4c5d-8216-1ee82a334cc6'), AIMessage(content="It seems you haven't performed any previous calculation. Could you clarify which number you'd like me to multiply by 5?", additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'ff3a1086-58fa-4eff-a375-53ee9d189293', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 10 Nov 2026 00:35:17 GMT', 'content-type': 'application/json', 'content-length': '323', 'connection': 'keep-alive', 'x-amzn-requestid': 'ff3a1086-58fa-4eff-a375-53ee9d189293'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [439]}, 'model_provider': 'bedrock_converse', 'model_name': 'mistral.mistral-large-3-675b-instruct'}, id='lc_run--01a1c2e1-6eb2-7901-8d87-3936db313061-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 453, 'output_tokens': 26, 'total_tokens'

`calc-2` is a new thread with no history, so the agent is as lost as the no-memory agent. Memory is stored **per thread**: one thread per user or chat session.

---
## Summary

| | Without memory | With memory |
|---|---|---|
| `create_agent(...)` | no `checkpointer` | `checkpointer=InMemorySaver()` |
| `stream/invoke` config | none | `{"configurable": {"thread_id": "..."}}` |
| "Multiply that by 5" | doesn't know what "that" is | uses the previous answer |
| Messages sent to the model | only the current question | the whole thread so far |

`InMemorySaver` lives in RAM, so restarting the kernel erases it. For storage that survives restarts, use `SqliteSaver` or `PostgresSaver`.